# Replication Workflow

Single working notebook for the Segnon and Trede replication. Keep durable logic in `src/`; use this notebook for exploration, diagnostics, and figures.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import (
    # config
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    ensure_project_dirs,

    # data
    load_price_csv,
    log_returns,
    align_return_frame,

    # statistics / backtesting
    summary_statistics,
    format_table_1,
    christoffersen_lr_test,
    violation_rate,

    # VaR v2 paper-like rolling functions
    SUPPORTED_COPULAS,
    RollingSpec,
    prepare_bivariate_returns,
    forecast_historical_var_rolling,
    forecast_variance_covariance_var_rolling,
    forecast_riskmetrics_var_rolling,
    forecast_ccc_garch_var_rolling,
    forecast_garch_copula_var_rolling,
    forecast_msm_copula_var_rolling,
    forecast_all_var_models,
    var_v2_portfolio_returns,
    var_v2_var_exceedances,
    violation_frequency,

    # plots
    plot_var_forecasts,
)

ensure_project_dirs()

## 1. Prices

In [2]:
raw_data_names = ["nasdaqcom_yahoo_close.csv", "sp500_yahoo_close.csv"]
raw_data_paths = [RAW_DATA_DIR / name for name in raw_data_names]

nasdaq_prices = load_price_csv(raw_data_paths[0])
sp500_prices = load_price_csv(raw_data_paths[1])
nasdaq_prices.name = "NASDAQ"
sp500_prices.name = "S&P 500"

prices = pd.concat(
    {
        "NASDAQ": nasdaq_prices,
        "S&P 500": sp500_prices,
    },
    axis=1,
).dropna()
prices.index.name = "date"

prices.head(), prices.tail(), prices.shape

(                 NASDAQ     S&P 500
 date                               
 2009-04-15  1626.800049  852.059998
 2009-04-16  1670.439941  865.299988
 2009-04-17  1673.069946  869.599976
 2009-04-20  1608.209961  832.390015
 2009-04-21  1643.849976  850.080017,
                  NASDAQ      S&P 500
 date                                
 2015-10-06  4748.359863  1979.920044
 2015-10-07  4791.149902  1995.829956
 2015-10-08  4810.790039  2013.430054
 2015-10-09  4830.470215  2014.890015
 2015-10-12  4838.640137  2017.459961,
 (1636, 2))

In [ ]:
#prices.to_csv(PROCESSED_DATA_DIR / "prices_nasdaq_sp500.csv")

In [4]:
fig1 = plot_price_evolution(
    prices,
    output_path=REPORTS_DIR / "figures" / "figure_1_prices.html",
)

fig1.show()

In [ ]:
#fig1.write_image(REPORTS_DIR / "figures" / "figure_1_prices.png", scale=2)

## 2. Build returns

In [3]:
returns = align_return_frame(
    {
        "NASDAQ": 100 * log_returns(prices["NASDAQ"]),
        "S&P 500": 100 * log_returns(prices["S&P 500"]),
    }
)
returns.head(), returns.tail(), returns.shape

(              NASDAQ   S&P 500
 date                          
 2009-04-16  2.647210  1.541931
 2009-04-17  0.157320  0.495705
 2009-04-20 -3.953850 -4.373221
 2009-04-21  2.191930  2.102938
 2009-04-22  0.137996 -0.771132,
               NASDAQ   S&P 500
 date                          
 2015-10-06 -0.690479 -0.359469
 2015-10-07  0.897118  0.800352
 2015-10-08  0.409087  0.877978
 2015-10-09  0.408250  0.072485
 2015-10-12  0.168990  0.127466,
 (1635, 2))

In [ ]:
#returns.to_csv(PROCESSED_DATA_DIR / "returns_nasdaq_sp500.csv")

In [8]:
fig2 = plot_returns_and_squared_returns(
    returns,
    output_path=REPORTS_DIR / "figures" / "figure_2_returns_squared_returns.html",
)

fig2.show()

In [ ]:
# fig2.write_image(
#     REPORTS_DIR / "figures" / "figure_2_returns_squared_returns.png",
#     scale=2,
# )

## 3. Descriptive Statistics

In [4]:
stats = summary_statistics(returns)
stats

,count,Mean,Std,Skewness,Kurtosis,Hurst,Tail index,Arch(1),Arch(1) p-value,Arch(5),Arch(5) p-value,Arch(10),Arch(10) p-value,JB,JB p-value,ADF,ADF p-value
NASDAQ,1635.0,0.066668,1.139080,-0.393170,6.025142,0.443318,3.852579,62.352988,2.870955e-15,243.573465,1.314607e-50,292.429966,6.185258e-57,660.180948,4.400770e-144,24.799764,0.0
S&P 500,1635.0,0.052718,1.035471,-0.429081,6.742166,0.435120,3.615215,69.899290,6.241092e-17,305.883788,5.439829e-64,345.043371,4.488230e-68,996.403046,4.303558e-217,20.273255,0.0


In [5]:
table_1 = format_table_1(stats)
table_1

,Mean,Std,Skewness,Kurtosis,Hurst,Tail index,Arch(1),Arch(5),Arch(10),JB,ADF
NASDAQ,0.067,1.139,-0.393,6.025,0.443,3.853,62.353 (0.000),243.573 (0.000),292.430 (0.000),660.181 (0.000),24.800 (0.000)
S&P 500,0.053,1.035,-0.429,6.742,0.435,3.615,69.899 (0.000),305.884 (0.000),345.043 (0.000),996.403 (0.000),20.273 (0.000)


- tail index: Hill est très sensible au seuil => actuellement fraction=0.5 donc 80 observ dans queues => le réduire ?
- écart sur ADF: peut-être du à diff de choix de lag (p-value à 0 donc même conclusion)

In [ ]:
#table_1.to_csv(REPORTS_DIR / "tables" / "table_1_statistics.csv", index=True)

Volatilité conditionnelle annuelle

In [17]:
float(float(table_1.loc['NASDAQ']['Std']) * np.sqrt(252)), float(float(table_1.loc['S&P 500']['Std']) * np.sqrt(252))

(18.081064459815412, 16.430115641711108)

## 4. MSM

runtime = 41min

In [6]:
msm_table = fit_msm_grid(
    returns=returns,
    k_values=range(1, 8),
    n_starts=30,
    seed=123,
)
msm_table

Estimating MSM: asset=NASDAQ, k=1, n_starts=30
  start 1/30: x0=[1.5        1.13908017 2.         0.1       ]
    success=True, loglik=-2386.293, nit=9, nfev=70
  start 2/30: x0=[1.5        1.13908017 5.         0.2       ]
    success=True, loglik=-2386.293, nit=9, nfev=65
  start 3/30: x0=[ 1.4         1.13908017 10.          0.1       ]
    success=True, loglik=-2386.293, nit=10, nfev=65
  start 4/30: x0=[ 1.6         1.13908017 10.          0.2       ]
    success=True, loglik=-2386.293, nit=9, nfev=65
  start 5/30: x0=[ 1.3         1.13908017 20.          0.1       ]
    success=True, loglik=-2386.293, nit=11, nfev=70
  start 6/30: x0=[1.36645353 1.0272659  5.42513298 0.45135148]
    success=True, loglik=-2386.293, nit=14, nfev=135
  start 7/30: x0=[ 1.24072268  0.77442334 12.30695595  0.69617121]
    success=True, loglik=-2386.293, nit=16, nfev=125
  start 8/30: x0=[ 1.72943919  1.35872252 25.41418408  0.84695736]
    success=True, loglik=-2386.293, nit=12, nfev=115
  start 9/30:

,asset,k,mean_return,m0,sigma,b,gamma_1,gamma_k,log_likelihood,success,message,nobs
0,NASDAQ,1,0.066668,1.626379,1.304680,2.000000,4.091375e-02,0.040914,-2386.292750,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
1,NASDAQ,2,0.066668,1.549589,1.343466,16.260779,8.891087e-03,0.135170,-2354.025612,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
2,NASDAQ,3,0.066668,1.512055,1.334460,20.799781,5.595557e-03,0.911752,-2349.759985,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
3,NASDAQ,4,0.066668,1.511582,1.908756,20.890437,2.643599e-04,0.910224,-2350.558626,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
4,NASDAQ,5,0.066668,1.511223,1.551741,20.140847,1.407945e-05,0.901417,-2350.625955,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
5,NASDAQ,6,0.066668,1.512051,2.224445,21.717327,5.189026e-07,0.918470,-2350.876234,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
6,NASDAQ,7,0.066668,1.512521,1.811152,22.540539,1.981866e-08,0.925676,-2350.894984,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
7,S&P 500,1,0.052718,1.701427,1.097393,20.379304,7.262538e-02,0.072625,-2183.472284,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
8,S&P 500,2,0.052718,1.594092,1.227832,14.420167,8.744065e-03,0.118954,-2136.246971,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635
9,S&P 500,3,0.052718,1.549599,1.220385,23.750954,4.598325e-03,0.925721,-2134.752201,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,1635


In [7]:
loglik_pivot = msm_table.pivot(
    index="asset",
    columns="k",
    values="log_likelihood",
)

loglik_pivot

k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,-2386.292750,-2354.025612,-2349.759985,-2350.558626,-2350.625955,-2350.876234,-2350.894984
S&P 500,-2183.472284,-2136.246971,-2134.752201,-2135.584183,-2135.597714,-2135.823087,-2136.246669


In [8]:
for param in ["m0", "sigma", "b", "gamma_k"]:
    display(
        msm_table.pivot(
            index="asset",
            columns="k",
            values=param,
        )
    )

k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,1.626379,1.549589,1.512055,1.511582,1.511223,1.512051,1.512521
S&P 500,1.701427,1.594092,1.549599,1.549782,1.550183,1.549809,1.550197


k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,1.304680,1.343466,1.334460,1.908756,1.551741,2.224445,1.811152
S&P 500,1.097393,1.227832,1.220385,0.980843,1.462294,2.178529,3.252207


k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,2.000000,16.260779,20.799781,20.890437,20.140847,21.717327,22.540539
S&P 500,20.379304,14.420167,23.750954,24.702648,27.619572,25.012387,26.667435


k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,0.040914,0.135170,0.911752,0.910224,0.901417,0.918470,0.925676
S&P 500,0.072625,0.118954,0.925721,0.931781,0.949355,0.933988,0.944198


In [ ]:
#msm_table.to_csv(REPORTS_DIR / "tables" / "table_2_msm_estimates_raw.csv", index=False)

In [ ]:
msm_table = pd.read_csv(REPORTS_DIR / "tables" / "table_2_msm_estimates_raw.csv")

best k:
- NASDAQ = 3
- SP500 = 3

papier: SP500 best k = 4

### k=3 & k=4

we use the selected k in the paper

In [24]:
%%time

res_nasdaq_final = fit_msm(
    returns=returns["NASDAQ"],
    k=3,
    n_starts=30,
    seed=123,
)

res_sp500_final = fit_msm(
    returns=returns["S&P 500"],
    k=4,
    n_starts=30,
    seed=123,
)

  start 1/30: x0=[1.5        1.13908017 2.         0.1       ]
    success=True, loglik=-2355.671, nit=4, nfev=35
  start 2/30: x0=[1.5        1.13908017 5.         0.2       ]
    success=True, loglik=-2353.842, nit=8, nfev=65
  start 3/30: x0=[ 1.4         1.13908017 10.          0.1       ]
    success=True, loglik=-2355.813, nit=10, nfev=90
  start 4/30: x0=[ 1.6         1.13908017 10.          0.2       ]
    success=True, loglik=-2355.792, nit=6, nfev=60
  start 5/30: x0=[ 1.3         1.13908017 20.          0.1       ]
    success=True, loglik=-2354.930, nit=7, nfev=75
  start 6/30: x0=[1.5776463  0.74475455 7.54636434 0.19146578]
    success=True, loglik=-2354.246, nit=10, nfev=85
  start 7/30: x0=[ 1.22313413  1.60848884 27.79233594  0.27721419]
    success=True, loglik=-2355.302, nit=6, nfev=40
  start 8/30: x0=[ 1.67382819  1.69710722 15.97354911  0.24781708]
    success=True, loglik=-2354.873, nit=12, nfev=100
  start 9/30: x0=[ 1.67696912  0.92694125 22.5542511   0.6058443

In [25]:
msm_fit_results = {
    "NASDAQ": res_nasdaq_final,
    "S&P 500": res_sp500_final,
}

In [26]:
pit_msm = build_msm_pit_frame(
    returns=returns,
    fit_results=msm_fit_results,
)

pit_msm.head()

,NASDAQ,S&P 500
date,,
2009-04-16,0.968529,0.862457
2009-04-17,0.527258,0.635519
2009-04-20,0.011079,0.011669
2009-04-21,0.881390,0.838473
2009-04-22,0.517535,0.330242


In [ ]:
#pit_msm.to_csv(PROCESSED_DATA_DIR / "pit_msm.csv")

In [13]:
pit_msm = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm.csv",
    index_col="date",
    parse_dates=True,
)

### k=3

testing with empirical results

In [27]:
res_sp500_emp = fit_msm(
    returns=returns["S&P 500"],
    k=3,
    n_starts=30,
    seed=123,
)

  start 1/30: x0=[1.5        1.03547075 2.         0.1       ]
    success=True, loglik=-2139.144, nit=12, nfev=85
  start 2/30: x0=[1.5        1.03547075 5.         0.2       ]
    success=True, loglik=-2140.102, nit=7, nfev=60
  start 3/30: x0=[ 1.4         1.03547075 10.          0.1       ]
    success=True, loglik=-2136.720, nit=10, nfev=65
  start 4/30: x0=[ 1.6         1.03547075 10.          0.2       ]
    success=True, loglik=-2137.886, nit=7, nfev=65
  start 5/30: x0=[ 1.3         1.03547075 20.          0.1       ]
    success=True, loglik=-2134.866, nit=11, nfev=105
  start 6/30: x0=[1.5776463  0.67701254 7.54636434 0.19146578]
    success=True, loglik=-2136.834, nit=10, nfev=95
  start 7/30: x0=[ 1.22313413  1.46218256 27.79233594  0.27721419]
    success=True, loglik=-2137.798, nit=8, nfev=60
  start 8/30: x0=[ 1.67382819  1.54274031 15.97354911  0.24781708]
    success=True, loglik=-2137.092, nit=10, nfev=95
  start 9/30: x0=[ 1.67696912  0.84262775 22.5542511   0.60584

In [ ]:
msm_fit_results_emp = {
    "NASDAQ": res_nasdaq_final,
    "S&P 500": res_sp500_emp,
}
pit_msm_emp = build_msm_pit_frame(
    returns=returns,
    fit_results=msm_fit_results_emp,
)
#pit_msm_emp.to_csv(PROCESSED_DATA_DIR / "pit_msm_emp.csv")

In [14]:
pit_msm_emp = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm_emp.csv",
    index_col="date",
    parse_dates=True,
)

### k=5

runtime = 4min

In [14]:
%%time

res_nasdaq_5 = fit_msm(
    returns=returns["NASDAQ"],
    k=5,
    n_starts=30,
    seed=123,
)

res_sp500_5 = fit_msm(
    returns=returns["S&P 500"],
    k=5,
    n_starts=30,
    seed=123,
)

  start 1/30: x0=[1.5        1.13908017 2.         0.1       ]
    success=True, loglik=-2352.691, nit=17, nfev=110
  start 2/30: x0=[1.5        1.13908017 5.         0.2       ]
    success=True, loglik=-2353.886, nit=10, nfev=80
  start 3/30: x0=[ 1.4         1.13908017 10.          0.1       ]
    success=True, loglik=-2355.485, nit=7, nfev=60
  start 4/30: x0=[ 1.6         1.13908017 10.          0.2       ]
    success=True, loglik=-2355.486, nit=7, nfev=50
  start 5/30: x0=[ 1.3         1.13908017 20.          0.1       ]
    success=True, loglik=-2355.321, nit=14, nfev=130
  start 6/30: x0=[1.5776463  0.74475455 7.54636434 0.19146578]
    success=True, loglik=-2358.019, nit=9, nfev=70
  start 7/30: x0=[ 1.22313413  1.60848884 27.79233594  0.27721419]
    success=True, loglik=-2355.565, nit=10, nfev=60
  start 8/30: x0=[ 1.67382819  1.69710722 15.97354911  0.24781708]
    success=True, loglik=-2350.959, nit=10, nfev=95
  start 9/30: x0=[ 1.67696912  0.92694125 22.5542511   0.6058

In [15]:
msm_fit_results_5 = {
    "NASDAQ": res_nasdaq_5,
    "S&P 500": res_sp500_5,
}

In [16]:
msm_fit_results_5

{'NASDAQ': MSMFitResult(asset='NASDAQ', k=5, params=MSMParams(m0=1.5099515751842003, sigma=1.5463133751015135, b=17.437947396085725, gamma_k=0.8622081702652361, gamma_1=2.1434827838340276e-05, k=5), log_likelihood=-2350.787756317032, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', nobs=1635, mean_return=0.06666781606711197),
 'S&P 500': MSMFitResult(asset='S&P 500', k=5, params=MSMParams(m0=1.5497260977124216, sigma=0.7878931452264746, b=24.541856781486622, gamma_k=0.9305471868008084, gamma_1=7.352064150700777e-06, k=5), log_likelihood=-2136.2833052449214, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', nobs=1635, mean_return=0.052717896639002546)}

In [ ]:
pit_msm = build_msm_pit_frame(
    returns=returns,
    fit_results=msm_fit_results_5,
)
#pit_msm.to_csv(PROCESSED_DATA_DIR / "pit_msm_5.csv")

In [15]:
pit_msm_5 = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm_5.csv",
    index_col="date",
    parse_dates=True,
)

## 5. GARCH

In [18]:
%%time

garch_results = fit_garch_marginals(
    returns=returns,
    mean="Constant",
    dist="normal",
    rescale=False,
)
garch_results

CPU times: total: 688 ms
Wall time: 1.25 s


{'NASDAQ': GARCHFitResult(asset='NASDAQ', model_result=                     Constant Mean - GARCH Model Results                      
 Dep. Variable:                 NASDAQ   R-squared:                       0.000
 Mean Model:             Constant Mean   Adj. R-squared:                  0.000
 Vol Model:                      GARCH   Log-Likelihood:               -2368.82
 Distribution:                  Normal   AIC:                           4745.64
 Method:            Maximum Likelihood   BIC:                           4767.24
                                         No. Observations:                 1635
 Date:                Tue, May 19 2026   Df Residuals:                     1634
 Time:                        10:54:53   Df Model:                            1
                                 Mean Model                                
                  coef    std err          t      P>|t|    95.0% Conf. Int.
 -------------------------------------------------------------------------

In [19]:
garch_table = garch_results_table(garch_results)
garch_table

,asset,mean,mean_se,omega,omega_se,alpha,alpha_se,beta,beta_se,log_likelihood,...,Arch(5),Arch(5) p-value,Arch(10),Arch(10) p-value,Q(2),Q(2) p-value,Q(4),Q(4) p-value,Q(8),Q(8) p-value
0,NASDAQ,0.094821,0.023171,0.043995,0.011718,0.103481,0.020528,0.860258,0.024105,-2368.818971,...,8.185071,0.146327,10.492696,0.398381,0.037705,0.981324,3.775267,0.437274,4.959775,0.761867
1,S&P 500,0.073626,0.019626,0.034368,0.008225,0.125234,0.022874,0.841043,0.023353,-2161.970578,...,16.262379,0.006134,18.804659,0.042815,1.037366,0.595304,2.639331,0.619872,7.459761,0.487934


In [ ]:
#garch_table.to_csv(REPORTS_DIR / "tables" / "table_3_garch_estimates_raw.csv", index=False)

In [14]:
garch_table = pd.read_csv(REPORTS_DIR / "tables" / "table_3_garch_estimates_raw.csv")

In [21]:
table_3_garch = format_garch_table_3(garch_table)
table_3_garch

,omega,alpha,beta,Arch(1),Arch(5),Arch(10),Q(2),Q(4),Q(8)
asset,,,,,,,,,
NASDAQ,0.044 [0.012],0.103 [0.021],0.860 [0.024],1.740 (0.187),8.185 (0.146),10.493 (0.398),0.038 (0.981),3.775 (0.437),4.960 (0.762)
S&P 500,0.034 [0.008],0.125 [0.023],0.841 [0.023],4.222 (0.040),16.262 (0.006),18.805 (0.043),1.037 (0.595),2.639 (0.620),7.460 (0.488)


In [ ]:
#table_3_garch.to_csv(REPORTS_DIR / "tables" / "table_3_garch_estimates_formatted.csv")

In [16]:
table_3_garch = pd.read_csv(REPORTS_DIR / "tables" / "table_3_garch_estimates_formatted.csv")

### Conditional Vol

In [43]:
garch_volatility = build_garch_volatility_frame(garch_results)
garch_volatility

,NASDAQ,S&P 500
date,,
2009-04-16,1.771182,1.768811
2009-04-17,1.848471,1.713396
2009-04-20,1.727358,1.589261
2009-04-21,2.075341,2.152921
2009-04-22,2.050428,2.109120
...,...,...
2015-10-06,1.413941,1.298467
2015-10-07,1.351911,1.214854
2015-10-08,1.297253,1.158351


In [ ]:
#garch_volatility.to_csv(PROCESSED_DATA_DIR / "garch_conditional_volatility.csv")

In [99]:
garch_volatility = pd.read_csv(PROCESSED_DATA_DIR / "garch_conditional_volatility.csv", index_col="date", parse_dates=True)

### PIT GARCH

In [23]:
pit_garch = build_garch_pit_frame(garch_results)
pit_garch

,NASDAQ,S&P 500
date,,
2009-04-16,0.925217,0.796761
2009-04-17,0.513486,0.597291
2009-04-20,0.009543,0.002571
2009-04-21,0.843870,0.827054
2009-04-22,0.508400,0.344385
...,...,...
2015-10-06,0.289311,0.369362
2015-10-07,0.723561,0.725147
2015-10-08,0.595709,0.756282


In [24]:
pit_garch.describe()

,NASDAQ,S&P 500
count,1635.000000,1635.000000
mean,0.499149,0.498485
std,0.275217,0.273879
min,0.000041,0.000022
25%,0.301117,0.291500
50%,0.506885,0.499895
75%,0.720656,0.714563
max,0.997744,0.999363


In [25]:
pit_garch.isna().sum()

NASDAQ     0
S&P 500    0
dtype: int64

In [ ]:
#pit_garch.to_csv(PROCESSED_DATA_DIR / "pit_garch.csv")

In [85]:
pit_garch = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_garch.csv",
    index_col="date",
    parse_dates=True,
)

## 6. Copulas

### MSM specifications used below

We keep three MSM specifications:

1. `paper_like_table2`: NASDAQ \(k=3\), S&P 500 \(k=4\), close to the model selection in Table 2.
2. `empirical_best`: NASDAQ \(k=3\), S&P 500 \(k=3\), selected from our likelihood table.
3. `paper_like_var`: NASDAQ \(k=5\), S&P 500 \(k=5\), because Table 4 and the VaR section of Segnon and Trede fix the MSM volatility components at \(k=5\).

The main VaR backtest will start with the `paper_like_var` specification.

In [19]:
pit_msm = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm_5.csv",
    index_col="date",
    parse_dates=True,
)
pit_msm_emp = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm_emp.csv",
    index_col="date",
    parse_dates=True,
)
pit_garch = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_garch.csv",
    index_col="date",
    parse_dates=True,
)

### paperilike : k=5

In [28]:
%%time

copula_results = fit_copula_grid(
    pit_by_model={
        "MSM": pit_msm,
        "GARCH": pit_garch,
    }
)

c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:240: RuntimeWarning: overflow encountered in power
  
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:240: RuntimeWarning: overflow encountered in power
  
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:240: RuntimeWarning: overflow encountered in power
  
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:240: RuntimeWarning: overflow encountered in power
  
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:240: Runt

CPU times: total: 9.14 s
Wall time: 9.62 s


In [29]:
copula_table = copula_results_table(copula_results)
copula_table

,margin_model,copula,log_likelihood,aic,bic,nobs,success,message,rho,nu,theta,tau_u,tau_l
0,MSM,gaussian,1790.237713,-3578.475426,-3573.076028,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.945020,NaN,NaN,NaN,NaN
1,MSM,student,1816.377223,-3628.754447,-3617.955651,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.944521,5.628161,NaN,NaN,NaN
2,MSM,plackett,1681.462603,-3360.925206,-3355.525808,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,75.222744,NaN,NaN
3,MSM,clayton,1471.764013,-2941.528025,-2936.128627,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,4.612738,NaN,NaN
4,MSM,rotated_clayton,1505.239404,-3008.478808,-3003.079410,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,4.888786,NaN,NaN
5,MSM,sjc,1805.082698,-3606.165397,-3595.366601,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,NaN,0.848531,0.820338
6,MSM,frank,1532.348929,-3062.697858,-3057.298460,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,20.000000,NaN,NaN
7,MSM,gumbel,1758.985979,-3515.971958,-3510.572560,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,4.480543,NaN,NaN
8,MSM,rotated_gumbel,1728.424141,-3454.848283,-3449.448885,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,4.389183,NaN,NaN
9,GARCH,gaussian,1841.477032,-3680.954064,-3675.554666,1635,True,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL,0.945868,NaN,NaN,NaN,NaN


In [30]:
table_4 = format_copula_table_4(copula_table)
table_4

,margin_model,copula,parameters,Log(L),AIC,BIC
0,MSM,gaussian,rho=0.945,1790.238,-3578.475,-3573.076
1,MSM,student,"rho=0.945, nu=5.628",1816.377,-3628.754,-3617.956
2,MSM,plackett,theta=75.223,1681.463,-3360.925,-3355.526
3,MSM,clayton,theta=4.613,1471.764,-2941.528,-2936.129
4,MSM,rotated_clayton,theta=4.889,1505.239,-3008.479,-3003.079
5,MSM,sjc,"tau_u=0.849, tau_l=0.820",1805.083,-3606.165,-3595.367
6,MSM,frank,theta=20.000,1532.349,-3062.698,-3057.298
7,MSM,gumbel,theta=4.481,1758.986,-3515.972,-3510.573
8,MSM,rotated_gumbel,theta=4.389,1728.424,-3454.848,-3449.449
9,GARCH,gaussian,rho=0.946,1841.477,-3680.954,-3675.555


In [ ]:
# copula_table.to_csv(
#     REPORTS_DIR / "tables" / "table_4_copula_estimates_raw.csv",
#     index=False,
# )

In [ ]:
# table_4.to_csv(
#     REPORTS_DIR / "tables" / "table_4_copula_estimates_formatted.csv",
#     index=False,
# )

In [33]:
best_loglik = (
    copula_table
    .loc[copula_table.groupby("margin_model")["log_likelihood"].idxmax()]
    .sort_values("margin_model")
)
best_aic = (
    copula_table
    .loc[copula_table.groupby("margin_model")["aic"].idxmin()]
    .sort_values("margin_model")
)
best_bic = (
    copula_table
    .loc[copula_table.groupby("margin_model")["bic"].idxmin()]
    .sort_values("margin_model")
)

best_summary = pd.concat(
    {
        "Max Log(L)": best_loglik,
        "Min AIC": best_aic,
        "Min BIC": best_bic,
    },
    names=["criterion"]
).reset_index(level=0)
best_summary[
    ["criterion", "margin_model", "copula", "log_likelihood", "aic", "bic", "rho", "nu", "theta"]
]

,criterion,margin_model,copula,log_likelihood,aic,bic,rho,nu,theta
10,Max Log(L),GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Max Log(L),MSM,student,1816.377223,-3628.754447,-3617.955651,0.944521,5.628161,NaN
10,Min AIC,GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Min AIC,MSM,student,1816.377223,-3628.754447,-3617.955651,0.944521,5.628161,NaN
10,Min BIC,GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Min BIC,MSM,student,1816.377223,-3628.754447,-3617.955651,0.944521,5.628161,NaN


### empirical_best: k=3

In [76]:
%%time

copula_results_emp = fit_copula_grid(
    pit_by_model={
        "MSM": pit_msm_emp,
        "GARCH": pit_garch,
    }
)

c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:182: RuntimeWarning: divide by zero encountered in divide
  density = numerator / denominator
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:182: RuntimeWarning: divide by zero encountered in divide
  density = numerator / denominator
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:182: RuntimeWarning: divide by zero encountered in divide
  density = numerator / denominator
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:182: RuntimeWarning: divide by zero encountered in divide
  density = numerator / denominator
c:\U

CPU times: total: 5.36 s
Wall time: 5.49 s


In [77]:
copula_table_emp = copula_results_table(copula_results_emp)
table_4_emp = format_copula_table_4(copula_table_emp)
table_4_emp

,margin_model,copula,parameters,Log(L),AIC,BIC
0,MSM,gaussian,rho=0.946,1799.563,-3597.125,-3591.726
1,MSM,student,"rho=0.945, nu=5.576",1825.919,-3647.837,-3637.038
2,MSM,plackett,theta=75.673,1686.451,-3370.902,-3365.503
3,MSM,clayton,theta=4.657,1485.549,-2969.098,-2963.699
4,MSM,rotated_clayton,theta=4.903,1510.657,-3019.315,-3013.915
5,MSM,frank,theta=20.000,1537.477,-3072.953,-3067.554
6,MSM,gumbel,theta=4.494,1766.015,-3530.029,-3524.630
7,MSM,rotated_gumbel,theta=4.415,1740.900,-3479.799,-3474.400
8,GARCH,gaussian,rho=0.946,1841.477,-3680.954,-3675.555
9,GARCH,student,"rho=0.946, nu=17.533",1846.797,-3689.594,-3678.795


In [ ]:
# copula_table_emp.to_csv(
#     REPORTS_DIR / "tables" / "table_4_copula_estimates_raw_emp.csv",
#     index=False,
# )

In [ ]:
# table_4_emp.to_csv(
#     REPORTS_DIR / "tables" / "table_4_copula_estimates_formatted_emp.csv",
#     index=False,
# )

In [81]:
best_loglik_emp = (
    copula_table_emp
    .loc[copula_table_emp.groupby("margin_model")["log_likelihood"].idxmax()]
    .sort_values("margin_model")
)
best_aic_emp = (
    copula_table_emp
    .loc[copula_table_emp.groupby("margin_model")["aic"].idxmin()]
    .sort_values("margin_model")
)
best_bic_emp = (
    copula_table_emp
    .loc[copula_table_emp.groupby("margin_model")["bic"].idxmin()]
    .sort_values("margin_model")
)

best_summary = pd.concat(
    {
        "Max Log(L)": best_loglik_emp,
        "Min AIC": best_aic_emp,
        "Min BIC": best_bic_emp,
    },
    names=["criterion"]
).reset_index(level=0)
best_summary[
    ["criterion", "margin_model", "copula", "log_likelihood", "aic", "bic", "rho", "nu", "theta"]
]

,criterion,margin_model,copula,log_likelihood,aic,bic,rho,nu,theta
9,Max Log(L),GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Max Log(L),MSM,student,1825.918610,-3647.837221,-3637.038425,0.944905,5.575925,NaN
9,Min AIC,GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Min AIC,MSM,student,1825.918610,-3647.837221,-3637.038425,0.944905,5.575925,NaN
9,Min BIC,GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Min BIC,MSM,student,1825.918610,-3647.837221,-3637.038425,0.944905,5.575925,NaN


## 7. Portfolio VaR

Reproduction of VaR - paper methodolodgy:
- fixed estimation window of 1135 observations ;
- 500 out-of-sample predictions ;
- portfolio equaly weighted by `PI` ;
- one-day ahead VaRs (5% and 1%)

Convention used: 

Var as a quantile :
$$
P(r_{p,t} \leq VaR_t(\alpha) \mid \Omega_{t-1}) = \alpha
$$

Violation happens when :
$$
r_{p,t} < VaR_t(\alpha)
$$

### 7.1 Parameters

In [2]:
PI = 0.5
WEIGHTS = np.array([PI, 1.0 - PI])
N_OOS = 500
WINDOW_SIZE = 1135

ALPHA_5 = 0.05
ALPHA_1 = 0.01

VAR_OUTPUT_DIR = TABLES_DIR / "var_forecasts"
VAR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS, WINDOW_SIZE, N_OOS

(array([0.5, 0.5]), 1135, 500)

In [3]:
returns = pd.read_csv(
    PROCESSED_DATA_DIR / "returns_nasdaq_sp500.csv",
    index_col="date",
    parse_dates=True,
)
returns

,NASDAQ,S&P 500
date,,
2009-04-16,2.647210,1.541931
2009-04-17,0.157320,0.495705
2009-04-20,-3.953850,-4.373221
2009-04-21,2.191930,2.102938
2009-04-22,0.137996,-0.771132
...,...,...
2015-10-06,-0.690479,-0.359469
2015-10-07,0.897118,0.800352
2015-10-08,0.409087,0.877978


### 7.2 Portfolio with out-of-sample returns

In [4]:
returns_var = prepare_bivariate_returns(
    returns,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
)

portfolio_ret = var_v2_portfolio_returns(
    returns_var,
    weights=WEIGHTS,
)

oos_index = returns_var.index[WINDOW_SIZE:WINDOW_SIZE + N_OOS]
portfolio_ret_oos = portfolio_ret.loc[oos_index]

print("returns_var shape:", returns_var.shape)
print("OOS length:", len(portfolio_ret_oos))
print("OOS start:", portfolio_ret_oos.index.min())
print("OOS end:", portfolio_ret_oos.index.max())

assert len(portfolio_ret_oos) == N_OOS

returns_var shape: (1635, 2)
OOS length: 500
OOS start: 2013-10-17 00:00:00
OOS end: 2015-10-12 00:00:00


### 7.3 utils

In [5]:
def save_var(series: pd.Series, filename: str) -> Path:
    """Save one VaR forecast series."""
    path = VAR_OUTPUT_DIR / filename
    series.to_frame(name=series.name or "VaR").to_csv(path)
    print(f"Saved: {path}")
    return path


def load_var(filename: str, column_name: str | None = None) -> pd.Series:
    """Load one saved VaR forecast series."""
    path = VAR_OUTPUT_DIR / filename
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    series = frame.iloc[:, 0]
    if column_name is not None:
        series.name = column_name
    return series


def concat_var_series(series_by_name: dict[str, pd.Series]) -> pd.DataFrame:
    """Concatenate VaR forecasts and align on common dates."""
    frame = pd.concat(
        {name: series for name, series in series_by_name.items()},
        axis=1,
    ).dropna(how="any")
    
    assert len(frame) == N_OOS, f"Expected {N_OOS} observations, got {len(frame)}"
    return frame


def backtest_var_panel(var_panel: pd.DataFrame, alpha: float) -> pd.DataFrame:
    """Compute violation frequency and Christoffersen LR tests for a VaR panel."""
    rows = []

    for model in var_panel.columns:
        hits = var_v2_var_exceedances(portfolio_ret_oos, var_panel[model])
        lr = christoffersen_lr_test(hits, alpha=alpha)

        rows.append(
            {
                "model": model,
                "alpha": alpha,
                "violations": int(hits.sum()),
                "EFV": hits.mean(),
                "LR_uc": lr["uc_stat"],
                "p_uc": lr["uc_pvalue"],
                "LR_ind": lr["ind_stat"],
                "p_ind": lr["ind_pvalue"],
                "LR_cc": lr["cc_stat"],
                "p_cc": lr["cc_pvalue"],
            }
        )

    return pd.DataFrame(rows).set_index("model")

### 7.4 VaR 5% — benchmark models

Fast models:
- Historical Simulation ;
- Variance-Covariance ;
- RiskMetrics ;
- CCC-GARCH.

In [7]:
# Historical Simulation VaR 5%

var_hist_5 = forecast_historical_var_rolling(
    returns=returns_var,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
)

save_var(var_hist_5.rename("Historical"), "var_hist_5.csv")
var_hist_5.head()

Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\reports\tables\var_forecasts\var_hist_5.csv


2013-10-17   -1.894145
2013-10-18   -1.894145
2013-10-21   -1.894145
2013-10-22   -1.885447
2013-10-23   -1.885447
Name: HS_VaR_0.05, dtype: float64

In [8]:
# Variance-Covariance VaR 5%

var_vc_5 = forecast_variance_covariance_var_rolling(
    returns=returns_var,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=True,
)

save_var(var_vc_5.rename("Variance-Covariance"), "var_vc_5.csv")
var_vc_5.head()

Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\reports\tables\var_forecasts\var_vc_5.csv


2013-10-17   -1.828738
2013-10-18   -1.827642
2013-10-21   -1.827548
2013-10-22   -1.812499
2013-10-23   -1.811373
Name: VarCov_VaR_0.05, dtype: float64

In [9]:
# RiskMetrics VaR 5%

var_rm_5 = forecast_riskmetrics_var_rolling(
    returns=returns_var,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    lambda_=0.94,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=False,
)

save_var(var_rm_5.rename("RiskMetrics"), "var_rm_5.csv")
var_rm_5.head()

Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\reports\tables\var_forecasts\var_rm_5.csv


2013-10-17   -1.910840
2013-10-18   -1.870711
2013-10-21   -1.856529
2013-10-22   -1.800248
2013-10-23   -1.753104
Name: RiskMetrics_VaR_0.05, dtype: float64

In [10]:
# CCC-GARCH VaR 5%

var_ccc_5 = forecast_ccc_garch_var_rolling(
    returns=returns_var,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=True,
    verbose=True,
)

save_var(var_ccc_5.rename("CCC-GARCH"), "var_ccc_5.csv")
var_ccc_5.head()

CCC-GARCH alpha=0.05: 25/500, VaR=-1.1924
CCC-GARCH alpha=0.05: 50/500, VaR=-1.1154
CCC-GARCH alpha=0.05: 75/500, VaR=-2.0139
CCC-GARCH alpha=0.05: 100/500, VaR=-1.1234
CCC-GARCH alpha=0.05: 125/500, VaR=-1.8614
CCC-GARCH alpha=0.05: 150/500, VaR=-1.2264
CCC-GARCH alpha=0.05: 175/500, VaR=-0.8985
CCC-GARCH alpha=0.05: 200/500, VaR=-1.4178
CCC-GARCH alpha=0.05: 225/500, VaR=-0.8564
CCC-GARCH alpha=0.05: 250/500, VaR=-2.1387
CCC-GARCH alpha=0.05: 275/500, VaR=-0.9747
CCC-GARCH alpha=0.05: 300/500, VaR=-1.6548
CCC-GARCH alpha=0.05: 325/500, VaR=-1.6683
CCC-GARCH alpha=0.05: 350/500, VaR=-1.1502
CCC-GARCH alpha=0.05: 375/500, VaR=-1.0054
CCC-GARCH alpha=0.05: 400/500, VaR=-1.0938
CCC-GARCH alpha=0.05: 425/500, VaR=-1.1255
CCC-GARCH alpha=0.05: 450/500, VaR=-1.2423
CCC-GARCH alpha=0.05: 475/500, VaR=-2.9218
CCC-GARCH alpha=0.05: 500/500, VaR=-1.5973
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Appro

2013-10-17   -1.574050
2013-10-18   -1.516937
2013-10-21   -1.509214
2013-10-22   -1.432491
2013-10-23   -1.374955
Name: CCC_GARCH_VaR_0.05, dtype: float64

#### Copula-GARCH

In [ ]:
# Copula-GARCH Student VaR 5%

var_garch_student_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula="student",
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(var_garch_student_5.rename("Copula-GARCH Student"), "var_garch_student_5.csv")
var_garch_student_5.head()

In [ ]:
COPULA_TO_RUN = "gaussian"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_5.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_5.csv",
)

var_garch_extra_5.head()

In [6]:
COPULA_TO_RUN = "sjc"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_5.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_5.csv",
)

var_garch_extra_5.head()

c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users

GARCH-sjc alpha=0.05: 25/500, VaR=-1.1848


c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users

GARCH-sjc alpha=0.05: 50/500, VaR=-1.1092


c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users

...

c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users

GARCH-sjc alpha=0.05: 475/500, VaR=-2.8978


c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:241: RuntimeWarning: overflow encountered in power
  s = a ** (-gamma) + b ** (-gamma) - 1.0
c:\Users

GARCH-sjc alpha=0.05: 500/500, VaR=-1.5862
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\reports\tables\var_forecasts\var_garch_sjc_5.csv


2013-10-17   -1.564764
2013-10-18   -1.507716
2013-10-21   -1.500353
2013-10-22   -1.423910
2013-10-23   -1.366841
Name: CopulaGARCH_sjc_VaR_0.05, dtype: float64

In [ ]:
COPULA_TO_RUN = "clayton"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_5.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_5.csv",
)

var_garch_extra_5.head()

In [ ]:
COPULA_TO_RUN = "rotated_clayton"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_5.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_5.csv",
)

var_garch_extra_5.head()

In [ ]:
COPULA_TO_RUN = "sjc"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_5.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_5.csv",
)

var_garch_extra_5.head()

In [ ]:
COPULA_TO_RUN = "frank"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_5.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_5.csv",
)

var_garch_extra_5.head()

In [ ]:
COPULA_TO_RUN = "gumbel"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_5.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_5.csv",
)

var_garch_extra_5.head()

In [ ]:
COPULA_TO_RUN = "rotated_gumbel"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_5 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_5.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_5.csv",
)

var_garch_extra_5.head()

### 7.5 VaR 5% - Copula-MSM

In [11]:
# Copula-MSM Student VaR 5%

var_msm_student_5 = forecast_msm_copula_var_rolling(
    returns=returns_var,
    copula="student",
    alpha=ALPHA_5,
    weights=WEIGHTS,
    k=5,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    n_starts=10,
    seed=123,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(var_msm_student_5.rename("Copula-MSM Student"), "var_msm_student_5.csv")
var_msm_student_5.head()

MSM-student alpha=0.05: 10/500, VaR=-1.1386
MSM-student alpha=0.05: 20/500, VaR=-1.2786
MSM-student alpha=0.05: 30/500, VaR=-1.1103
MSM-student alpha=0.05: 40/500, VaR=-1.2532
MSM-student alpha=0.05: 50/500, VaR=-1.1323
MSM-student alpha=0.05: 60/500, VaR=-1.0251


KeyboardInterrupt: 

In [ ]:
COPULA_TO_RUN = "plackett"  # options: plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_msm_extra_5 = forecast_msm_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_5,
    weights=WEIGHTS,
    k=5,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    n_starts=10,
    seed=123,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_msm_extra_5.rename(f"Copula-MSM {COPULA_TO_RUN}"),
    f"var_msm_{COPULA_TO_RUN}_5.csv",
)

var_msm_extra_5.head()

### 7.6 VaR 5% - concat

In [7]:
var_series_fig = {
    "Historical": load_var("var_hist_5.csv"),
    "Variance-Covariance": load_var("var_vc_5.csv"),
    "RiskMetrics": load_var("var_rm_5.csv"),
    "CCC-GARCH": load_var("var_ccc_5.csv"),
    "Copula-GARCH Student": load_var("var_garch_student_5.csv"),
    "Copula-MSM Student": load_var("var_msm_student_5.csv"),
}

var_panel_fig = concat_var_series(var_series_fig)

var_panel_fig.to_csv(VAR_OUTPUT_DIR / "var_panel_fig.csv")

In [27]:
var_series_all = {
    "Historical": load_var("var_hist_5.csv"),
    "Variance-Covariance": load_var("var_vc_5.csv"),
    "RiskMetrics": load_var("var_rm_5.csv"),
    "CCC-GARCH": load_var("var_ccc_5.csv"),
    "Copula-GARCH Student": load_var("var_garch_student_5.csv"),
    "Copula-GARCH Gaussian": load_var("var_garch_gaussian_5.csv"),
    "Copula-GARCH Plackett": load_var("var_garch_plackett_5.csv"),
    "Copula-GARCH Clayton": load_var("var_garch_clayton_5.csv"),
    "Copula-GARCH Rotated Clayton": load_var("var_garch_rotated_clayton_5.csv"),
    "Copula-GARCH SJC": load_var("var_garch_sjc_5.csv"),
    "Copula-GARCH Frank": load_var("var_garch_frank_5.csv"),
    "Copula-GARCH Gumbel": load_var("var_garch_gumbel_5.csv"),
    "Copula-GARCH Rotated Gumbel": load_var("var_garch_rotated_gumbel_5.csv"),
    "Copula-MSM Student": load_var("var_msm_student_5.csv"),
    "Copula-MSM Gaussian": load_var("var_msm_gaussian_5.csv"),
    #"Copula-MSM Plackett": load_var("var_msm_plackett_5.csv"),
    #"Copula-MSM Clayton": load_var("var_msm_clayton_5.csv"),
    #"Copula-MSM Rotated Clayton": load_var("var_msm_rotated_clayton_5.csv"),
    #"Copula-MSM SJC": load_var("var_msm_sjc_5.csv"),
    #"Copula-MSM Frank": load_var("var_msm_frank_5.csv"),
    #"Copula-MSM Gumbel": load_var("var_msm_gumbel_5.csv"),
    #"Copula-MSM Rotated Gumbel": load_var("var_msm_rotated_gumbel_5.csv"),
}

var_panel_all = concat_var_series(var_series_all)

var_panel_all.to_csv(VAR_OUTPUT_DIR / "var_panel_all.csv")
var_panel_all.head()

,Historical,Variance-Covariance,RiskMetrics,CCC-GARCH,Copula-GARCH Student,Copula-GARCH Gaussian,Copula-GARCH Plackett,Copula-GARCH Clayton,Copula-GARCH Rotated Clayton,Copula-GARCH SJC,Copula-GARCH Frank,Copula-GARCH Gumbel,Copula-GARCH Rotated Gumbel,Copula-MSM Student,Copula-MSM Gaussian
2013-10-17,-1.894145,-1.828738,-1.910840,-1.574050,-1.574052,-1.574028,-1.562749,-1.584488,-1.470044,-1.564764,-1.571929,-1.554903,-1.581432,-1.644574,-1.645104
2013-10-18,-1.894145,-1.827642,-1.870711,-1.516937,-1.516933,-1.516911,-1.506014,-1.527016,-1.416833,-1.507716,-1.514875,-1.498525,-1.524069,-1.594554,-1.595078
2013-10-21,-1.894145,-1.827548,-1.856529,-1.509214,-1.509214,-1.509196,-1.498351,-1.519248,-1.409575,-1.500353,-1.505908,-1.490878,-1.516312,-1.626070,-1.626556
2013-10-22,-1.885447,-1.812499,-1.800248,-1.432491,-1.432539,-1.432501,-1.422304,-1.442232,-1.337011,-1.423910,-1.440224,-1.414939,-1.439409,-1.512771,-1.515012
2013-10-23,-1.885447,-1.811373,-1.753104,-1.374955,-1.375004,-1.374967,-1.365100,-1.384396,-1.282619,-1.366841,-1.376855,-1.357950,-1.381657,-1.436967,-1.438793


### 7.7 VaR 5% - Graph

In [8]:
fig3 = plot_var_forecasts(
    portfolio_returns=portfolio_ret_oos.rename("Portfolio returns"),
    var_forecasts=var_panel_fig,
    output_path=REPORTS_DIR / "figures" / "figure_3_var_5pct.html",
    title="Figure 3 — VaR forecasts at the 5% confidence level",
    positive_loss_var=False,
)

fig3.show()

In [9]:
fig3.write_image(
    REPORTS_DIR / "figures" / "figure_3_var_5pct.png",
    scale=2,
)

### 7.8 VaR 1% - benchmark models

In [ ]:
var_hist_1 = forecast_historical_var_rolling(
    returns=returns_var,
    alpha=ALPHA_1,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
)
save_var(var_hist_1.rename("Historical"), "var_hist_1.csv")

var_vc_1 = forecast_variance_covariance_var_rolling(
    returns=returns_var,
    alpha=ALPHA_1,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=True,
)
save_var(var_vc_1.rename("Variance-Covariance"), "var_vc_1.csv")

var_rm_1 = forecast_riskmetrics_var_rolling(
    returns=returns_var,
    alpha=ALPHA_1,
    weights=WEIGHTS,
    lambda_=0.94,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=False,
)
save_var(var_rm_1.rename("RiskMetrics"), "var_rm_1.csv")

var_ccc_1 = forecast_ccc_garch_var_rolling(
    returns=returns_var,
    alpha=ALPHA_1,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=True,
    verbose=True,
)
save_var(var_ccc_1.rename("CCC-GARCH"), "var_ccc_1.csv")


#### Copula-GARCH

In [ ]:
var_garch_student_1 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula="student",
    alpha=ALPHA_1,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)
save_var(var_garch_student_1.rename("Copula-GARCH Student"), "var_garch_student_1.csv")
var_garch_student_1.head()

In [ ]:
COPULA_TO_RUN = "gaussian"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_extra_1 = forecast_garch_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_1,
    weights=WEIGHTS,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_garch_extra_1.rename(f"Copula-GARCH {COPULA_TO_RUN}"),
    f"var_garch_{COPULA_TO_RUN}_1.csv",
)

var_garch_extra_1.head()

### 7.9 VaR 1% - Copula-MSM

In [ ]:
var_msm_student_1 = forecast_msm_copula_var_rolling(
    returns=returns_var,
    copula="student",
    alpha=ALPHA_1,
    weights=WEIGHTS,
    k=5,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    n_starts=10,
    seed=123,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(var_msm_student_1.rename("Copula-MSM Student"), "var_msm_student_1.csv")
var_msm_student_1.head()

In [6]:
COPULA_TO_RUN = "gaussian"  # options: gaussian, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_msm_extra_1 = forecast_msm_copula_var_rolling(
    returns=returns_var,
    copula=COPULA_TO_RUN,
    alpha=ALPHA_1,
    weights=WEIGHTS,
    k=5,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
    n_starts=10,
    seed=123,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

save_var(
    var_msm_extra_1.rename(f"Copula-MSM {COPULA_TO_RUN}"),
    f"var_msm_{COPULA_TO_RUN}_1.csv",
)

var_msm_extra_1.head()

MSM-gaussian alpha=0.01: 10/500, VaR=-1.9559, x1=[ 1.5245  1.6205 22.7343  0.9332], x2=[ 1.5464  1.5025 24.5116  0.9309]


KeyboardInterrupt: 

### 7.10 VaR 1% - concat

In [10]:
var_series_fig_1 = {
    "Historical": load_var("var_hist_1.csv"),
    "Variance-Covariance": load_var("var_vc_1.csv"),
    "RiskMetrics": load_var("var_rm_1.csv"),
    "CCC-GARCH": load_var("var_ccc_1.csv"),
    "Copula-GARCH Student": load_var("var_garch_student_1.csv"),
    "Copula-MSM Student": load_var("var_msm_student_1.csv"),
}

var_panel_fig_1 = concat_var_series(var_series_fig_1)

var_panel_fig_1.to_csv(VAR_OUTPUT_DIR / "var_panel_fig_1.csv")

In [28]:
var_series_all_1 = {
    "Historical": load_var("var_hist_1.csv"),
    "Variance-Covariance": load_var("var_vc_1.csv"),
    "RiskMetrics": load_var("var_rm_1.csv"),
    "CCC-GARCH": load_var("var_ccc_1.csv"),
    "Copula-GARCH Student": load_var("var_garch_student_1.csv"),
    "Copula-GARCH Gaussian": load_var("var_garch_gaussian_1.csv"),
    "Copula-GARCH Plackett": load_var("var_garch_plackett_1.csv"),
    "Copula-GARCH Clayton": load_var("var_garch_clayton_1.csv"),
    "Copula-GARCH Rotated Clayton": load_var("var_garch_rotated_clayton_1.csv"),
    "Copula-GARCH SJC": load_var("var_garch_sjc_1.csv"),
    "Copula-GARCH Frank": load_var("var_garch_frank_1.csv"),
    "Copula-GARCH Gumbel": load_var("var_garch_gumbel_1.csv"),
    "Copula-GARCH Rotated Gumbel": load_var("var_garch_rotated_gumbel_1.csv"),
    "Copula-MSM Student": load_var("var_msm_student_1.csv"),
    #"Copula-MSM Gaussian": load_var("var_msm_gaussian_1.csv"),
    #"Copula-MSM Plackett": load_var("var_msm_plackett_1.csv"),
    #"Copula-MSM Clayton": load_var("var_msm_clayton_1.csv"),
    #"Copula-MSM Rotated Clayton": load_var("var_msm_rotated_clayton_1.csv"),
    #"Copula-MSM SJC": load_var("var_msm_sjc_1.csv"),
    #"Copula-MSM Frank": load_var("var_msm_frank_1.csv"),
    #"Copula-MSM Gumbel": load_var("var_msm_gumbel_1.csv"),
    #"Copula-MSM Rotated Gumbel": load_var("var_msm_rotated_gumbel_1.csv"),
}

var_panel_all_1 = concat_var_series(var_series_all_1)

var_panel_all_1.to_csv(VAR_OUTPUT_DIR / "var_panel_all_1.csv")
var_panel_all_1.head()

,Historical,Variance-Covariance,RiskMetrics,CCC-GARCH,Copula-GARCH Student,Copula-GARCH Gaussian,Copula-GARCH Plackett,Copula-GARCH Clayton,Copula-GARCH Rotated Clayton,Copula-GARCH SJC,Copula-GARCH Frank,Copula-GARCH Gumbel,Copula-GARCH Rotated Gumbel,Copula-MSM Student
2013-10-17,-3.211284,-2.614929,-2.702538,-2.266403,-2.268438,-2.266370,-2.208520,-2.288371,-1.970638,-2.275947,-2.124393,-2.221528,-2.285164,-2.595488
2013-10-18,-3.211284,-2.612849,-2.645782,-2.185638,-2.187684,-2.185600,-2.129779,-2.206803,-1.900353,-2.194750,-2.048471,-2.142452,-2.203715,-2.524178
2013-10-21,-3.211284,-2.612957,-2.625725,-2.175097,-2.177101,-2.175068,-2.119550,-2.196156,-1.891765,-2.184293,-2.036731,-2.132114,-2.193081,-2.567240
2013-10-22,-3.111556,-2.593221,-2.546125,-2.067385,-2.069479,-2.067397,-2.014603,-2.087758,-1.796775,-2.076445,-1.956249,-2.026223,-2.084802,-2.534799
2013-10-23,-3.111556,-2.590992,-2.479448,-1.985968,-1.987988,-1.985983,-1.934955,-2.005695,-1.724852,-1.994552,-1.867816,-1.946130,-2.002829,-2.442562


### 7.11 VaR 1% - Graph

In [11]:
fig4 = plot_var_forecasts(
    portfolio_returns=portfolio_ret_oos.rename("Portfolio returns"),
    var_forecasts=var_panel_fig_1,
    output_path=REPORTS_DIR / "figures" / "figure_4_var_1pct.html",
    title="Figure 4 — VaR forecasts at the 1% confidence level",
    positive_loss_var=False,
)

fig4.show()

In [12]:
fig4.write_image(
    REPORTS_DIR / "figures" / "figure_4_var_1pct.png",
    scale=2,
)

## 8. LR test

In [29]:
from src.risk import christoffersen_lr_test, var_exceedances
from src.var_v2 import portfolio_returns

def make_table_lr(returns, var_panel, alpha, weights=(0.5, 0.5)):
    rp = portfolio_returns(returns, weights)
    rp = rp.reindex(var_panel.index)

    rows = {}

    for model in var_panel.columns:
        hits = var_exceedances(rp, var_panel[model])
        test = christoffersen_lr_test(hits, alpha)

        rows[model] = {
            "EFV": hits.mean(),
            "uc": test["uc_pvalue"],
            "ind": test["ind_pvalue"],
            "cc": test["cc_pvalue"],
        }

    return pd.DataFrame(rows).T

In [41]:
def parse_model_name(model):
    if model in ["Historical", "RiskMetrics", "Variance-Covariance", "CCC-GARCH"]:
        group = "Bench"
        submodel = {
            "Historical": "Historical",
            "RiskMetrics": "RiskMetrics",
            "Variance-Covariance": "Var-Covar",
            "CCC-GARCH": "CCC-GARCH",
        }[model]
    elif model.startswith("Copula-GARCH"):
        group = "Copula-GARCH"
        submodel = model.replace("Copula-GARCH ", "")
    elif model.startswith("Copula-MSM"):
        group = "Copula-MSM"
        submodel = model.replace("Copula-MSM ", "")
    else:
        group = "Other"
        submodel = model

    submodel = submodel.replace("Gaussian", "Normal")
    submodel = submodel.replace("Rotated Clayton", "rotClayton")
    submodel = submodel.replace("Rotated Gumbel", "rotGumbel")

    return group, submodel

### 8.1 LR test: VaR 5% forecast

In [66]:
table_5 = make_table_lr(returns, var_panel_all, alpha=0.05)
table_5

,EFV,uc,ind,cc
Historical,0.036,0.131347,0.155388,0.116773
Variance-Covariance,0.040,0.288479,0.044012,0.074915
RiskMetrics,0.072,0.033677,0.791915,0.101234
CCC-GARCH,0.072,0.033677,0.678951,0.096215
Copula-GARCH Student,0.072,0.033677,0.678951,0.096215
Copula-GARCH Gaussian,0.072,0.033677,0.678951,0.096215
Copula-GARCH Plackett,0.072,0.033677,0.678951,0.096215
Copula-GARCH Clayton,0.070,0.052333,0.748039,0.144550
Copula-GARCH Rotated Clayton,0.076,0.012912,0.550230,0.038045
Copula-GARCH SJC,0.072,0.033677,0.678951,0.096215


In [ ]:
#table_5.to_csv(REPORTS_DIR / "tables" / "table_5_lr_var_5pct_raw.csv", index=True)

In [67]:
table_5 = table_5.reset_index().rename(columns={"index": "model"})
table_5

,model,EFV,uc,ind,cc
0,Historical,0.036,0.131347,0.155388,0.116773
1,Variance-Covariance,0.040,0.288479,0.044012,0.074915
2,RiskMetrics,0.072,0.033677,0.791915,0.101234
3,CCC-GARCH,0.072,0.033677,0.678951,0.096215
4,Copula-GARCH Student,0.072,0.033677,0.678951,0.096215
5,Copula-GARCH Gaussian,0.072,0.033677,0.678951,0.096215
6,Copula-GARCH Plackett,0.072,0.033677,0.678951,0.096215
7,Copula-GARCH Clayton,0.070,0.052333,0.748039,0.144550
8,Copula-GARCH Rotated Clayton,0.076,0.012912,0.550230,0.038045
9,Copula-GARCH SJC,0.072,0.033677,0.678951,0.096215


In [68]:
df_long = table_5.melt(
    id_vars="model",
    value_vars=["EFV", "uc", "ind", "cc"],
    var_name="stat",
    value_name="value"
)
df_long[["group", "submodel"]] = df_long["model"].apply(
    lambda x: pd.Series(parse_model_name(x))
)

table_5_formatted = df_long.pivot(
    index="stat",
    columns=["group", "submodel"],
    values="value"
)
table_5_formatted

group         Bench                                 Copula-GARCH            \
submodel Historical Var-Covar RiskMetrics CCC-GARCH      Student    Normal   
stat                                                                         
EFV        0.036000  0.040000    0.072000  0.072000     0.072000  0.072000   
cc         0.116773  0.074915    0.101234  0.096215     0.096215  0.096215   
ind        0.155388  0.044012    0.791915  0.678951     0.678951  0.678951   
uc         0.131347  0.288479    0.033677  0.033677     0.033677  0.033677   

group                                                                  \
submodel  Plackett   Clayton rotClayton       SJC     Frank    Gumbel   
stat                                                                    
EFV       0.072000  0.070000   0.076000  0.072000  0.070000  0.072000   
cc        0.096215  0.144550   0.038045  0.096215  0.144550  0.096215   
ind       0.678951  0.748039   0.550230  0.678951  0.748039  0.678951   
uc        0.033677  0.052333   0.012912  0.033677  0.052333  0.033677   

group              Copula-MSM            
submodel rotGumbel    Student    Normal  
stat                                     
EFV       0.070000   0.072000  0.072000  
cc        0.144550   0.096215  0.096215  
ind       0.748039   0.678951  0.678951  
uc        0.052333   0.033677  0.033677

In [69]:
stat_order = ["EFV", "uc", "ind", "cc"]
bench_order = ["Historical", "RiskMetrics", "Var-Covar", "CCC-GARCH"]
copula_order = [
    "Normal", "Student", "Plackett", "Clayton",
    "rotClayton", "SJC", "Frank", "Gumbel", "rotGumbel"
]
column_order = (
    [("Bench", c) for c in bench_order]
    + [("Copula-GARCH", c) for c in copula_order]
    + [("Copula-MSM", c) for c in copula_order[:2]]
)
table_5_formatted = table_5_formatted.reindex(index=stat_order, columns=pd.MultiIndex.from_tuples(column_order))
table_5_formatted = table_5_formatted.map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
table_5_formatted

Bench                                 Copula-GARCH                   \
     Historical RiskMetrics Var-Covar CCC-GARCH       Normal Student Plackett   
stat                                                                            
EFV       0.036       0.072     0.040     0.072        0.072   0.072    0.072   
uc        0.131       0.034     0.288     0.034        0.034   0.034    0.034   
ind       0.155       0.792     0.044     0.679        0.679   0.679    0.679   
cc        0.117       0.101     0.075     0.096        0.096   0.096    0.096   

                                                       Copula-MSM          
     Clayton rotClayton    SJC  Frank Gumbel rotGumbel     Normal Student  
stat                                                                       
EFV    0.070      0.076  0.072  0.070  0.072     0.070      0.072   0.072  
uc     0.052      0.013  0.034  0.052  0.034     0.052      0.034   0.034  
ind    0.748      0.550  0.679  0.748  0.679     0.748      0.679   0.679  
cc     0.145      0.038  0.096  0.145  0.096     0.145      0.096   0.096

In [ ]:
# table_5_formatted.to_csv(
#     REPORTS_DIR / "tables" / "table_5_lr_var_5pct_formatted.csv",
#     index=True,
#     encoding="utf-8-sig"
# )

In [65]:
table_5_formatted = pd.read_csv(REPORTS_DIR / "tables" / "table_5_lr_var_5pct_formatted.csv", header=[0, 1],
    index_col=0
)
table_5_formatted

Bench                                 Copula-GARCH                   \
     Historical RiskMetrics Var-Covar CCC-GARCH       Normal Student Plackett   
stat                                                                            
EFV       0.036       0.072     0.040     0.072        0.072   0.072    0.072   
uc        0.131       0.034     0.288     0.034        0.034   0.034    0.034   
ind       0.155       0.792     0.044     0.679        0.679   0.679    0.679   
cc        0.117       0.101     0.075     0.096        0.096   0.096    0.096   

                                                       Copula-MSM          
     Clayton rotClayton    SJC  Frank Gumbel rotGumbel     Normal Student  
stat                                                                       
EFV    0.070      0.076  0.072  0.070  0.072     0.070      0.072   0.072  
uc     0.052      0.013  0.034  0.052  0.034     0.052      0.034   0.034  
ind    0.748      0.550  0.679  0.748  0.679     0.748      0.679   0.679  
cc     0.145      0.038  0.096  0.145  0.096     0.145      0.096   0.096

In [ ]:
latex = table_5_formatted.to_latex(
    index=True,
    multicolumn=True,
    multicolumn_format="c",
    multirow=False,
    escape=False,
    caption="The results of LR test using VaR(5%) forecasts.",
    label="tab:lr_var_5",
)

with open(REPORTS_DIR / "tables" / "table_5_lr_var_5pct.tex", "w", encoding="utf-8") as f:
    f.write(latex)

### 8.2 LR test: VaR 1% forecast

In [79]:
table_6 = make_table_lr(returns, var_panel_all_1, alpha=0.01)
table_6

,EFV,uc,ind,cc
Historical,0.008,6.414349e-01,0.019432,5.845258e-02
Variance-Covariance,0.014,3.965697e-01,0.002146,6.286847e-03
RiskMetrics,0.032,8.395385e-05,0.096153,1.097107e-04
CCC-GARCH,0.030,2.857199e-04,0.073289,2.788692e-04
Copula-GARCH Student,0.030,2.857199e-04,0.073289,2.788692e-04
Copula-GARCH Gaussian,0.030,2.857199e-04,0.073289,2.788692e-04
Copula-GARCH Plackett,0.030,2.857199e-04,0.073289,2.788692e-04
Copula-GARCH Clayton,0.030,2.857199e-04,0.073289,2.788692e-04
Copula-GARCH Rotated Clayton,0.046,3.540639e-09,0.388719,1.856326e-08
Copula-GARCH SJC,0.030,2.857199e-04,0.073289,2.788692e-04


In [72]:
table_6.to_csv(REPORTS_DIR / "tables" / "table_6_lr_var_1pct_raw.csv")

In [80]:
table_6 = table_6.reset_index().rename(columns={"index": "model"})

df_long = table_6.melt(
    id_vars="model",
    value_vars=["EFV", "uc", "ind", "cc"],
    var_name="stat",
    value_name="value"
)
df_long[["group", "submodel"]] = df_long["model"].apply(
    lambda x: pd.Series(parse_model_name(x))
)
table_6_formatted = df_long.pivot(
    index="stat",
    columns=["group", "submodel"],
    values="value"
)

column_order = (
    [("Bench", c) for c in bench_order]
    + [("Copula-GARCH", c) for c in copula_order]
    + [("Copula-MSM", "Student")]
)
table_6_formatted = table_6_formatted.reindex(index=stat_order, columns=pd.MultiIndex.from_tuples(column_order))
table_6_formatted = table_6_formatted.map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
table_6_formatted

Bench                                 Copula-GARCH                   \
     Historical RiskMetrics Var-Covar CCC-GARCH       Normal Student Plackett   
stat                                                                            
EFV       0.008       0.032     0.014     0.030        0.030   0.030    0.030   
uc        0.641       0.000     0.397     0.000        0.000   0.000    0.000   
ind       0.019       0.096     0.002     0.073        0.073   0.073    0.073   
cc        0.058       0.000     0.006     0.000        0.000   0.000    0.000   

                                                       Copula-MSM  
     Clayton rotClayton    SJC  Frank Gumbel rotGumbel    Student  
stat                                                               
EFV    0.030      0.046  0.030  0.036  0.030     0.030      0.014  
uc     0.000      0.000  0.000  0.000  0.000     0.000      0.397  
ind    0.073      0.389  0.073  0.155  0.073     0.073      0.002  
cc     0.000      0.000  0.000  0.000  0.000     0.000      0.006

In [ ]:
# table_6_formatted.to_csv(
#     REPORTS_DIR / "tables" / "table_6_lr_var_1pct_formatted.csv",
#     index=True,
#     encoding="utf-8-sig"
# )

In [ ]:
table_6_formatted = pd.read_csv(REPORTS_DIR / "tables" / "table_6_lr_var_1pct_formatted.csv", header=[0, 1],
    index_col=0
)

In [82]:
latex = table_6_formatted.to_latex(
    index=True,
    multicolumn=True,
    multicolumn_format="c",
    multirow=False,
    escape=False,
    caption="The results of LR test using VaR(1%) forecasts.",
    label="tab:lr_var_1",
)

with open(REPORTS_DIR / "tables" / "table_6_lr_var_1pct.tex", "w", encoding="utf-8") as f:
    f.write(latex)

## OLD VaR

### Pre-backtest

In [129]:
hits_5pct = var_exceedances(
    returns=portfolio_oos,
    var_forecasts=var_forecasts["Student-Copula-MSM VaR 5%"],
)
hits_1pct = var_exceedances(
    returns=portfolio_oos,
    var_forecasts=var_forecasts["Student-Copula-MSM VaR 1%"],
)

pre_backtest_summary = pd.DataFrame(
    {
        "alpha": [0.05, 0.01],
        "nobs": [len(hits_5pct), len(hits_1pct)],
        "violations": [hits_5pct.sum(), hits_1pct.sum()],
        "efv": [hits_5pct.mean(), hits_1pct.mean()],
    },
    index=["VaR 5%", "VaR 1%"],
)

pre_backtest_summary

,alpha,nobs,violations,efv
VaR 5%,0.05,500,35,0.070
VaR 1%,0.01,500,6,0.012
